# 1 import & const

In [ ]:
from pathlib import Path

musicChannelsFilePath = Path(r"C:\02Programmer\02Proj\PyVSCode\ConfigPrivate\Rep001Tools002YTProjV4\musicChannels.txt")

In [ ]:
def readFileLine(path: Path, line_number: int = 0, strip_newline: bool = True) -> list[str]:
    try:
        with path.open('r', encoding='utf-8') as f:
            if line_number <= 0:
                return [line.strip() for line in f if line.strip()]
            else:
                for current_line_num, line in enumerate(f, start=1):
                    if current_line_num == line_number:
                        return [line.rstrip('\n') if strip_newline else line]
    except FileNotFoundError:
        print(f"文件未找到: {path}")
    except Exception as e:
        print(f"读取文件时出错: {e}")
    return []

readFileLine(musicChannelsFilePath, 1)

In [ ]:
def wait_for_user_input():
    """程序暂停，等待用户输入'y'继续或'n'退出"""
    while True:
        user_input = input("输入 'y' 继续，输入 'n' 退出: ")
        if user_input.lower() == 'n':
            return False  # 用户选择退出，返回 False
        elif user_input.lower() == 'y':
            return True  # 用户选择继续，返回 True
        else:
            print("无效输入，请输入 'y' 或 'n'。")
            
# wait_for_user_input()

In [ ]:
# 从 Firefox 获取 cookies 并 格式化为 Netscape 格式（yt-dlp 支持的格式）
import time
import browser_cookie3


def reget_cookies(cookies_path: Path):
    # 从 Firefox 获取 cookies
    cookies = browser_cookie3.firefox(domain_name='youtube.com')

    # 格式化为 Netscape 格式（yt-dlp 支持的格式）
    cookies_txt_path = cookies_path

    with open(cookies_txt_path, 'w', encoding='utf-8') as f:
        f.write("# Netscape HTTP Cookie File\n")
        for cookie in cookies:
            domain = cookie.domain if cookie.domain.startswith('.') else '.' + cookie.domain
            path = cookie.path or '/'
            secure = "TRUE" if cookie.secure else "FALSE"
            expires = int(time.time()) + 3600 * 24 * 30  # 设置过期时间为 30 天
            f.write(f"{domain}\tTRUE\t{path}\t{secure}\t{expires}\t{cookie.name}\t{cookie.value}\n")
    print(f"Cookies 已保存到: {cookies_txt_path}")
    
# reget_cookies(cookies_path)

In [ ]:
from yt_dlp import YoutubeDL

line_num = 241
output_root = Path(r"C:\02Programmer\02Proj\PyVSCode\PyGitRep001\Tools\005YTProjMerge\music2")   # 输出根目录
cookies_path = Path(r"C:\02Programmer\02Proj\PyVSCode\ConfigPrivate\Rep001Tools002YTProjV4\cookies.txt")     # 用于身份验证的 cookies 路径

while True:
    # 读取下一行
    line = readFileLine(musicChannelsFilePath, line_number=line_num)
    if not line:
        print("已经是文件末尾,取不出来链接了。")
        # 如果没有更多行，暂停并等待用户输入，让用户增加更多行
        if not wait_for_user_input():  # 如果用户选择退出，停止程序
            print("没有更多内容，程序终止。")
            break
        # 如果用户输入'y'，继续
        print("继续处理下一行...")
        continue  # 继续处理下一行

    print(line)

    # 设置下载选项
    ydl_opts = {
        # 'cookiesfrombrowser': ('firefox',),   # 从Firefox提取cookie
        'cookies': cookies_path,  # 使用 cookies 文件
        'quiet': True,            # 不输出任何日志
        'extract_flat': True,     # 提取所有视频链接而不下载
        'force_generic_extractor': True,  # 强制使用通用提取器
        'outtmpl': '%(id)s.%(ext)s'  # 不需要实际保存文件
    }

    # 创建 yt-dlp 下载器实例
    with YoutubeDL(ydl_opts) as ydl:
        # 获取播放列表中的所有视频信息
        result = ydl.extract_info(line[0], download=False)

    # 提取视频链接
    if 'entries' in result:
        urls = [entry['url'] for entry in result['entries']]
        
    print(urls)


    ydl_opts = {
        # 自动判断媒体类型
        'format': 'bestaudio/best/bestvideo+bestaudio/best',                      # 优先音频流，若无则退回best
        'ignoreerrors': True,       # 出错跳过
        'noplaylist': False,        # 支持整页下载
        'continuedl': True,         # 支持断点续传
        'no_warnings': True,        # 屏蔽无关警告
        
        'quiet': True,          # ✅ 静默模式（不打印控制台输出）
        'no_warnings': True,    # ✅ 不显示警告信息
        'verbose': False,       # ✅ 不输出调试信息
        'logger': None,         # ✅ 禁用 logger
    
        'outtmpl': str(output_root) + '/line_num_' + str(line_num) + '/%(uploader)s/%(album)s/%(upload_date)s-%(title)s-%(view_count)s.%(ext)s',     # 保存路径模板

        # ⚙️ 输出模板
        # 若为音频：Music/歌手/专辑/标题.mp3
        # 若为视频：Video/作者/日期-播放量-标题.mp4
        # 'outtmpl': {
        #     'default': str(output_root / 'Video' / '%(uploader)s' / '%(upload_date)s-%(title)s-%(view_count_fmt)s.%(ext)s'),
        #     'bestaudio': str(output_root / 'Music' / '%(artist)s' / '%(album)s' / '%(title)s.%(ext)s'),
        # },

        'extractor_args': {
            'youtube': {
                'skip': ['authcheck'],
            }
        },
        'ignoreerrors': True,
    
        # ✅ 显示可用格式列表时不报错
        # 注意：listformats=True 只列出，不下载
        # 下载前不需要再设置这个，否则它只会打印格式然后退出
        # 'listformats': True,   ← ⚠️ 请注释掉这一行
    
        'writethumbnail': True,               # 下载缩略图（封面）
        'writeinfojson': True,                # 保存 info.json 文件
        'encoding': 'utf-8',                  # 防止乱码
        'concurrent_fragment_downloads': 3,   # 同时下载3个分片
        'fragment_retries': 5,                # 自动重试分片下载失败
        'rejecttitle': 'Live',                # 排除标题包含“Live”的视频
        
        # 💾 下载与转换设置
        'merge_output_format': 'mp3',         # 若音频则输出 mp3
        'postprocessors': [
            {'key': 'FFmpegExtractAudio', 'preferredcodec': 'mp3', 'preferredquality': '0'},
            {'key': 'EmbedThumbnail'},        # 嵌入封面
            {'key': 'FFmpegMetadata'},        # 写入元数据
        ],

        # 🧩 Cookie 设置
        'cookiesfrombrowser': ('firefox',),   # 从Firefox提取cookie
        'cookies': cookies_path,  # 使用 cookies 文件

        # 🧾 下载记录文件
        'download_archive': str(output_root / 'musicDownloaded.txt'),
        
        # 🕓 网络与稳定性
        'retries': 3,                         # 下载重试次数
        'fragment_retries': 3,
        'socket_timeout': 30,
        'sleep_interval': 3,                  # 每个视频间隔
        'max_sleep_interval': 10,             # 随机间隔

        # 🔒 文件名冲突处理
        'overwrites': False,                  # 不覆盖已存在文件
    }
    
    for url in urls:
        try:
            # 创建 yt-dlp 下载器实例
            with YoutubeDL(ydl_opts) as ydl:
                ydl.download(url)
        except Exception as e:
            # 捕获与 cookies 相关的错误（比如 cookies 失效）
            if "Sign in to confirm you're not a bot" in str(e):
                # 如果遇到 cookie 错误，重新获取 cookies 并重试
                reget_cookies(cookies_path)
                # 更新配置文件中的 cookies 路径
                ydl_opts['cookies'] = cookies_path  # 更新为新获得的 cookies 文件路径
                # 重新下载
                ydl.download(url)
            else:
                # 如果是其他错误，则打印错误信息
                print(f"下载失败: {e}")

    print(f"下载完第 {line_num} 行。")
    line_num += 1  # 更新行号